# Market Trend History — Data Cleaning Notebook
**Dataset:** Job Postings Salary Prediction (Kaggle) — 50.000 postings, 2020–2024  
**Output:** `market_trend_history.csv` — format time-series bulanan per skill per domain  
**Tujuan:** Input model Prophet/LSTM untuk prediksi tren skill di Job Market Dashboard


## Step 0 — Import Library

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
import warnings
warnings.filterwarnings('ignore')


## Step 1 — Load Data
Load file `job_postings.csv` dan lihat gambaran awal dataset.


In [2]:
df = pd.read_csv('archive/job_postings.csv')

print(f"Shape: {df.shape}")
print(f"\nKolom: {df.columns.tolist()}")
print(f"\nRentang tanggal: {df['post_date'].min()} s/d {df['post_date'].max()}")
df.head(3)


Shape: (50000, 24)

Kolom: ['job_id', 'post_date', 'year', 'month', 'job_title', 'industry', 'company', 'company_size', 'location', 'state', 'work_type', 'experience_level', 'experience_min_yrs', 'experience_max_yrs', 'education_required', 'salary_min', 'salary_max', 'salary_midpoint', 'skills_required', 'benefits', 'days_to_fill', 'applicant_count', 'seniority', 'is_tech_role']

Rentang tanggal: 2020-01-01 s/d 2024-12-31


,job_id,post_date,year,month,job_title,industry,company,company_size,location,state,...,education_required,salary_min,salary_max,salary_midpoint,skills_required,benefits,days_to_fill,applicant_count,seniority,is_tech_role
0,J0000001,2023-12-30,2023,12,Quantitative Analyst,Education,Summit Digital,Enterprise (5000+),Remote,Remote,...,Bachelor's,126000,153000,139000,Python|Financial Modeling|Derivatives,Dental/Vision|Stock Options|401k,29,41,Junior,0
1,J0000002,2022-05-25,2022,5,Backend Developer,Education,Nexus AI,Large (1001-5000),"Boston, MA",MA,...,Bachelor's,297000,432000,362000,Redis|Docker|Python|Java|AWS,Health Insurance|Flexible Hours|Remote Work|De...,20,88,Executive,0
2,J0000003,2020-09-08,2020,9,HR Manager,Consulting,Summit Corp,Mid-size (201-1000),"Atlanta, GA",GA,...,PhD,118000,157000,136000,Compensation|Talent Acquisition|Employment Law,Flexible Hours|Stock Options|Parental Leave,33,52,Senior,0


In [3]:
print("=== Tipe Data ===")
print(df.dtypes)
print("\n=== Missing Values ===")
print(df.isnull().sum())


=== Tipe Data ===
job_id                  str
post_date               str
year                  int64
month                 int64
job_title               str
industry                str
company                 str
company_size            str
location                str
state                   str
work_type               str
experience_level        str
experience_min_yrs    int64
experience_max_yrs    int64
education_required      str
salary_min            int64
salary_max            int64
salary_midpoint       int64
skills_required         str
benefits                str
days_to_fill          int64
applicant_count       int64
seniority               str
is_tech_role          int64
dtype: object

=== Missing Values ===
job_id                0
post_date             0
year                  0
month                 0
job_title             0
industry              0
company               0
company_size          0
location              0
state                 0
work_type             0
experien

## Step 2 — Hapus Kolom yang Tidak Relevan
Dari 24 kolom yang ada, kita hanya butuh kolom yang relevan untuk membuat time-series tren skill.

Kolom yang **dihapus**:
- `is_tech_role` → tidak digunakan karena nilainya tidak akurat (per instruksi tim AI). Filter IT job dilakukan lewat keyword `job_title` di Step 3
- `benefits`, `days_to_fill`, `applicant_count` → tidak relevan untuk analisis tren skill
- `salary_min`, `salary_max`, `salary_midpoint` → tidak dibutuhkan untuk output akhir
- `company`, `location`, `state` → terlalu granular, tidak dibutuhkan
- `seniority`, `experience_level`, `experience_min_yrs`, `experience_max_yrs` → bukan fokus dataset ini
- `education_required`, `work_type`, `industry`, `company_size` → tidak dibutuhkan
- `year`, `month` → akan di-derive ulang dari `post_date`
- `job_id` → tidak dibutuhkan setelah agregasi


In [4]:
cols_to_keep = ['post_date', 'job_title', 'skills_required']

df = df[cols_to_keep].copy()
print(f"Kolom tersisa: {df.columns.tolist()}")
print(f"Shape: {df.shape}")
df.head(3)


Kolom tersisa: ['post_date', 'job_title', 'skills_required']
Shape: (50000, 3)


,post_date,job_title,skills_required
0,2023-12-30,Quantitative Analyst,Python|Financial Modeling|Derivatives
1,2022-05-25,Backend Developer,Redis|Docker|Python|Java|AWS
2,2020-09-08,HR Manager,Compensation|Talent Acquisition|Employment Law


## Step 3 — Filter Hanya IT Jobs (Berbasis Keyword `job_title`)
Dataset ini berisi campuran job IT dan non-IT (HR Manager, Financial Analyst, Marketing Manager, dll).  
Kolom `is_tech_role` **tidak digunakan** karena tidak akurat.  

Sebagai gantinya, kita filter berdasarkan **keyword pada `job_title`** — job yang jelas masuk kategori IT dipertahankan, sisanya dihapus.


In [5]:
# Daftar job title yang termasuk IT
IT_JOB_TITLES = {
    'Frontend Developer',
    'Backend Developer',
    'Software Engineer',
    'Senior Software Engineer',
    'Data Scientist',
    'Machine Learning Engineer',
    'Data Analyst',
    'Data Engineer',
    'DevOps Engineer',
    'Cloud Architect',
    'Cybersecurity Analyst',
    'UX Designer',
    'Product Manager',
    'Business Analyst',
    'Research Scientist'
}

# Cek semua job title yang ada di dataset
print("Semua job title di dataset:")
for title in sorted(df['job_title'].unique()):
    status = "✓ IT" if title in IT_JOB_TITLES else "✗ Non-IT (dihapus)"
    print(f"  {status:20s} | {title}")


Semua job title di dataset:
  ✓ IT                 | Backend Developer
  ✓ IT                 | Business Analyst
  ✓ IT                 | Cloud Architect
  ✓ IT                 | Cybersecurity Analyst
  ✓ IT                 | Data Analyst
  ✓ IT                 | Data Engineer
  ✓ IT                 | Data Scientist
  ✓ IT                 | DevOps Engineer
  ✗ Non-IT (dihapus)   | Financial Analyst
  ✓ IT                 | Frontend Developer
  ✗ Non-IT (dihapus)   | HR Manager
  ✓ IT                 | Machine Learning Engineer
  ✗ Non-IT (dihapus)   | Marketing Manager
  ✗ Non-IT (dihapus)   | Operations Manager
  ✓ IT                 | Product Manager
  ✗ Non-IT (dihapus)   | Quantitative Analyst
  ✓ IT                 | Research Scientist
  ✓ IT                 | Senior Software Engineer
  ✓ IT                 | Software Engineer
  ✓ IT                 | UX Designer


In [6]:
# Filter hanya IT jobs
before = len(df)
df = df[df['job_title'].isin(IT_JOB_TITLES)].copy()
after = len(df)

print(f"Sebelum filter : {before:,} baris")
print(f"Setelah filter : {after:,} baris")
print(f"Dihapus        : {before - after:,} baris (non-IT jobs)")
print(f"\nDistribusi job_title setelah filter:")
print(df['job_title'].value_counts())


Sebelum filter : 50,000 baris
Setelah filter : 37,476 baris
Dihapus        : 12,524 baris (non-IT jobs)

Distribusi job_title setelah filter:
job_title
Research Scientist           2589
Business Analyst             2576
Senior Software Engineer     2550
Cybersecurity Analyst        2540
Machine Learning Engineer    2536
Backend Developer            2508
Cloud Architect              2500
Data Scientist               2497
DevOps Engineer              2497
Data Analyst                 2490
UX Designer                  2457
Software Engineer            2444
Data Engineer                2442
Product Manager              2442
Frontend Developer           2408
Name: count, dtype: int64


## Step 4 — Mapping `job_title` → `job_domain`
Map setiap IT job title ke domain yang sesuai dengan master job catalog.


In [7]:
TITLE_TO_DOMAIN = {
    'Frontend Developer':         'Web Development',
    'Backend Developer':          'Web Development',
    'Software Engineer':          'Software Development',
    'Senior Software Engineer':   'Software Development',
    'Data Scientist':             'Data Science & AI',
    'Machine Learning Engineer':  'Data Science & AI',
    'Research Scientist':         'Data Science & AI',      
    'Data Analyst':               'Data Analytics',
    'Business Analyst':           'Data Analytics',         
    'Data Engineer':              'Data Engineering',
    'DevOps Engineer':            'DevOps & Cloud',
    'Cloud Architect':            'DevOps & Cloud',
    'Cybersecurity Analyst':      'Cyber Security',
    'UX Designer':                'UI/UX Design',
    'Product Manager':            'Product Management',
}

df['job_domain'] = df['job_title'].map(TITLE_TO_DOMAIN)

# Validasi tidak ada yang tidak ter-map
unmapped = df[df['job_domain'].isnull()]['job_title'].unique()
if len(unmapped) == 0:
    print("Semua job title berhasil di-map ke domain. ✓")
else:
    print(f"Job title tidak ter-map: {unmapped}")

print(f"\nDistribusi job_domain:")
print(df['job_domain'].value_counts())


Semua job title berhasil di-map ke domain. ✓

Distribusi job_domain:
job_domain
Data Science & AI       7622
Data Analytics          5066
DevOps & Cloud          4997
Software Development    4994
Web Development         4916
Cyber Security          2540
UI/UX Design            2457
Data Engineering        2442
Product Management      2442
Name: count, dtype: int64


## Step 5 — Cleaning Kolom `post_date`
Konversi ke datetime dan set `date_recorded` = hari terakhir tiap bulan (konvensi standar time-series bulanan).


In [8]:
df['post_date'] = pd.to_datetime(df['post_date'])

invalid_dates = df['post_date'].isnull().sum()
print(f"Tanggal invalid/null: {invalid_dates}")

df['date_recorded'] = df['post_date'] + pd.offsets.MonthEnd(0)

print(f"\nRentang date_recorded: {df['date_recorded'].min().date()} s/d {df['date_recorded'].max().date()}")
print(f"Jumlah bulan unik: {df['date_recorded'].nunique()}")
print(f"\nSample bulan:")
print(sorted(df['date_recorded'].dt.to_period('M').unique())[:12])


Tanggal invalid/null: 0

Rentang date_recorded: 2020-01-31 s/d 2024-12-31
Jumlah bulan unik: 60

Sample bulan:
[Period('2020-01', 'M'), Period('2020-02', 'M'), Period('2020-03', 'M'), Period('2020-04', 'M'), Period('2020-05', 'M'), Period('2020-06', 'M'), Period('2020-07', 'M'), Period('2020-08', 'M'), Period('2020-09', 'M'), Period('2020-10', 'M'), Period('2020-11', 'M'), Period('2020-12', 'M')]


## Step 6 — Cleaning & Standardisasi `skills_required`
Kolom ini berisi skill dipisah `|`. Langkah:
1. Split per skill
2. Strip whitespace
3. Standardisasi nama agar konsisten dengan master job catalog


In [9]:
print("Sample skills_required:")
for s in df['skills_required'].head(5):
    print(f"  {s}")

all_skills_raw = Counter()
for val in df['skills_required'].dropna():
    for s in val.split('|'):
        all_skills_raw[s.strip()] += 1

print(f"\nTotal unique skills (sebelum standardisasi): {len(all_skills_raw)}")
print(f"\nTop 20 skills:")
for skill, count in all_skills_raw.most_common(20):
    print(f"  {count:6d}  {skill}")


Sample skills_required:
  Redis|Docker|Python|Java|AWS
  Terraform|GCP|Microservices
  Git|Kubernetes|Java|Docker|Python|REST APIs
  Java|Docker|AWS|Python|CI/CD
  dbt|Snowflake|AWS|Spark|Python

Total unique skills (sebelum standardisasi): 57

Top 20 skills:
   14997  Python
   10952  SQL
    9805  AWS
    7070  Docker
    6983  Kubernetes
    4953  Tableau
    4635  Statistics
    4506  TensorFlow
    4282  REST APIs
    4280  Git
    4247  Java
    4165  Spark
    3743  Agile
    3724  Stakeholder Management
    3494  Excel
    3377  Figma
    3064  R
    3015  Machine Learning
    2859  CI/CD
    2838  Microservices


In [10]:
SKILL_STANDARDIZE = {
    'REST APIs':    'REST API',
    'HTML/CSS':     'HTML',
    'Roadmapping':  'Roadmap',
    'Analytics':    'Data Analysis',
    'Research':     'User Research',
}

def clean_skills(skills_str):
    """Split, strip, dan standardisasi nama skill."""
    if pd.isnull(skills_str):
        return []
    skills = [s.strip() for s in skills_str.split('|')]
    skills = [SKILL_STANDARDIZE.get(s, s) for s in skills]
    return skills

df['skills_list'] = df['skills_required'].apply(clean_skills)

print("Sample hasil cleaning:")
for skills in df['skills_list'].head(5):
    print(f"  {skills}")


Sample hasil cleaning:
  ['Redis', 'Docker', 'Python', 'Java', 'AWS']
  ['Terraform', 'GCP', 'Microservices']
  ['Git', 'Kubernetes', 'Java', 'Docker', 'Python', 'REST API']
  ['Java', 'Docker', 'AWS', 'Python', 'CI/CD']
  ['dbt', 'Snowflake', 'AWS', 'Spark', 'Python']


## Step 7 — Explode Skills & Agregasi Bulanan
Pecah satu baris per skill, lalu hitung `demand_count` = jumlah posting yang membutuhkan skill tersebut per domain per bulan.


In [11]:
df_exploded = df[['date_recorded', 'job_domain', 'skills_list']].explode('skills_list')
df_exploded = df_exploded.rename(columns={'skills_list': 'skill_name'})

df_exploded = df_exploded[df_exploded['skill_name'].notna()]
df_exploded = df_exploded[df_exploded['skill_name'] != '']

print(f"Shape setelah explode: {df_exploded.shape}")
print(f"\nSample:")
print(df_exploded.head(8).to_string(index=False))


Shape setelah explode: (167834, 3)

Sample:
date_recorded      job_domain    skill_name
   2022-05-31 Web Development         Redis
   2022-05-31 Web Development        Docker
   2022-05-31 Web Development        Python
   2022-05-31 Web Development          Java
   2022-05-31 Web Development           AWS
   2022-01-31  DevOps & Cloud     Terraform
   2022-01-31  DevOps & Cloud           GCP
   2022-01-31  DevOps & Cloud Microservices


In [12]:
df_agg = (df_exploded
          .groupby(['date_recorded', 'job_domain', 'skill_name'])
          .size()
          .reset_index(name='demand_count'))

df_agg['date_recorded'] = df_agg['date_recorded'].dt.strftime('%Y-%m-%d')

print(f"Shape setelah agregasi: {df_agg.shape}")
print(f"\nSample:")
print(df_agg.head(8).to_string(index=False))


Shape setelah agregasi: (5340, 4)

Sample:
date_recorded     job_domain          skill_name  demand_count
   2020-01-31 Cyber Security          Compliance            30
   2020-01-31 Cyber Security   Incident Response            35
   2020-01-31 Cyber Security    Network Security            36
   2020-01-31 Cyber Security Penetration Testing            35
   2020-01-31 Cyber Security              Python            29
   2020-01-31 Cyber Security                SIEM            33
   2020-01-31 Data Analytics               Agile            24
   2020-01-31 Data Analytics  Data Visualization            20


## Step 8 — Filter Skill Non-Teknis
Hapus skill yang tidak relevan dengan dunia IT (skill HR, Finance, Marketing, Operations).


In [13]:
SKILLS_TO_REMOVE = {
    # HR / GA
    'Compensation', 'Talent Acquisition', 'Performance Management', 'HRIS', 'Employment Law',
    
    # Finance / Accounting
    'Financial Modeling', 'Derivatives', 'FP&A', 'Bloomberg', 'Accounting', 'Budgeting',
    
    # Operations / General Management
    'Supply Chain', 'Lean', 'Six Sigma', 'Process Improvement', 'Stakeholder Management',
    
    # Marketing
    'Content Strategy', 'SEO', 'Google Ads'
}

before = len(df_agg)
df_agg = df_agg[~df_agg['skill_name'].isin(SKILLS_TO_REMOVE)]
after = len(df_agg)

print(f"Sebelum filter skill : {before:,} baris")
print(f"Setelah filter skill : {after:,} baris")
print(f"Dihapus              : {before - after:,} baris")
print(f"\nSkill unik tersisa: {df_agg['skill_name'].nunique()}")
print(f"\nDaftar skill yang dipertahankan:")
print(sorted(df_agg['skill_name'].unique()))


Sebelum filter skill : 5,340 baris
Setelah filter skill : 5,220 baris
Dihapus              : 120 baris

Skill unik tersisa: 55

Daftar skill yang dipertahankan:
['A/B Testing', 'AWS', 'Adobe XD', 'Agile', 'Airflow', 'Azure', 'CI/CD', 'Compliance', 'Cost Optimization', 'Data Visualization', 'Design Thinking', 'Docker', 'Excel', 'Figma', 'GCP', 'Git', 'HTML', 'Incident Response', 'Java', 'JavaScript', 'Kafka', 'Kubernetes', 'Linux', 'MLOps', 'Machine Learning', 'Microservices', 'Network Security', 'Penetration Testing', 'PostgreSQL', 'Power BI', 'Product Strategy', 'Prototyping', 'Publications', 'PyTorch', 'Python', 'R', 'REST API', 'React', 'Redis', 'Requirements Gathering', 'Roadmap', 'SIEM', 'SQL', 'Security', 'Snowflake', 'Spark', 'Statistics', 'System Design', 'Tableau', 'TensorFlow', 'Terraform', 'TypeScript', 'Usability Testing', 'User Research', 'dbt']


## Step 9 — Validasi Data

In [14]:
print("=== VALIDASI AKHIR ===")
print(f"Shape final        : {df_agg.shape}")
print(f"Kolom              : {df_agg.columns.tolist()}")
print(f"Missing values     :\n{df_agg.isnull().sum()}")
print(f"Duplikat           : {df_agg.duplicated(subset=['date_recorded','job_domain','skill_name']).sum()}")
print(f"Rentang tanggal    : {df_agg['date_recorded'].min()} s/d {df_agg['date_recorded'].max()}")
print(f"Jumlah bulan unik  : {df_agg['date_recorded'].nunique()}")
print(f"\nDistribusi skill per domain:")
print(df_agg.groupby('job_domain')['skill_name'].nunique().sort_values(ascending=False))
print(f"\nTop 10 skill berdasarkan total demand:")
print(df_agg.groupby('skill_name')['demand_count'].sum().sort_values(ascending=False).head(10))


=== VALIDASI AKHIR ===
Shape final        : (5220, 4)
Kolom              : ['date_recorded', 'job_domain', 'skill_name', 'demand_count']
Missing values     :
date_recorded    0
job_domain       0
skill_name       0
demand_count     0
dtype: int64
Duplikat           : 0
Rentang tanggal    : 2020-01-31 s/d 2024-12-31
Jumlah bulan unik  : 60

Distribusi skill per domain:
job_domain
Data Science & AI       15
Web Development         14
DevOps & Cloud          13
Software Development    11
Data Analytics           9
Data Engineering         8
Cyber Security           6
UI/UX Design             6
Product Management       5
Name: skill_name, dtype: int64

Top 10 skill berdasarkan total demand:
skill_name
Python        14997
SQL           10952
AWS            9805
Docker         7070
Kubernetes     6983
Tableau        4953
Statistics     4635
TensorFlow     4506
REST API       4282
Git            4280
Name: demand_count, dtype: int64


In [15]:
# Cek coverage bulan per skill 
skill_month_count = df_agg.groupby('skill_name')['date_recorded'].nunique()
print("Coverage bulan per skill:")
print(skill_month_count.describe())

kurang = skill_month_count[skill_month_count < 50]
if len(kurang) > 0:
    print(f"\nSkill dengan coverage < 50 bulan:")
    print(kurang.sort_values())
else:
    print("\nSemua skill memiliki coverage bulan yang baik (≥50 bulan). ✓")


Coverage bulan per skill:
count    55.0
mean     60.0
std       0.0
min      60.0
25%      60.0
50%      60.0
75%      60.0
max      60.0
Name: date_recorded, dtype: float64

Semua skill memiliki coverage bulan yang baik (≥50 bulan). ✓


## Step 10 — Simpan ke CSV

In [16]:
df_final = df_agg.sort_values(['job_domain', 'skill_name', 'date_recorded']).reset_index(drop=True)

print("=== PREVIEW FINAL ===")
print(df_final.head(10).to_string(index=False))
print(f"\nTotal baris: {len(df_final):,}")

df_final.to_csv('market_trend_history.csv', index=False)
print("\n✓ Berhasil disimpan sebagai market_trend_history.csv")


=== PREVIEW FINAL ===
date_recorded     job_domain skill_name  demand_count
   2020-01-31 Cyber Security Compliance            30
   2020-02-29 Cyber Security Compliance            25
   2020-03-31 Cyber Security Compliance            36
   2020-04-30 Cyber Security Compliance            24
   2020-05-31 Cyber Security Compliance            30
   2020-06-30 Cyber Security Compliance            28
   2020-07-31 Cyber Security Compliance            38
   2020-08-31 Cyber Security Compliance            24
   2020-09-30 Cyber Security Compliance            29
   2020-10-31 Cyber Security Compliance            40

Total baris: 5,220

✓ Berhasil disimpan sebagai market_trend_history.csv


In [17]:
df_check = pd.read_csv('market_trend_history.csv')
print(f"Verifikasi file tersimpan:")
print(f"  Shape  : {df_check.shape}")
print(f"  Kolom  : {df_check.columns.tolist()}")
print(f"\nContoh — Python di Data Science & AI:")
print(df_check[(df_check['skill_name']=='Python') & 
               (df_check['job_domain']=='Data Science & AI')].head(6).to_string(index=False))


Verifikasi file tersimpan:
  Shape  : (5220, 4)
  Kolom  : ['date_recorded', 'job_domain', 'skill_name', 'demand_count']

Contoh — Python di Data Science & AI:
date_recorded        job_domain skill_name  demand_count
   2020-01-31 Data Science & AI     Python            66
   2020-02-29 Data Science & AI     Python            59
   2020-03-31 Data Science & AI     Python            78
   2020-04-30 Data Science & AI     Python            66
   2020-05-31 Data Science & AI     Python            71
   2020-06-30 Data Science & AI     Python            67


## Ringkasan Proses Cleaning

| Step | Proses | Hasil |
|------|--------|-------|
| 1 | Load data | 50.000 baris, 24 kolom |
| 2 | Hapus kolom tidak relevan | Tersisa: `post_date`, `job_title`, `skills_required` |
| 3 | **Filter IT jobs via keyword `job_title`** | 50.000 → ~32.500 baris (hapus HR, Finance, Marketing, dll) |
| 4 | Mapping `job_title` → `job_domain` | 13 IT job title → 9 domain |
| 5 | Cleaning `post_date` → `date_recorded` | Hari terakhir bulan, 60 bulan (Jan 2020–Des 2024) |
| 6 | Standardisasi nama skill | Konsisten dengan master job catalog |
| 7 | Explode & agregasi bulanan | `demand_count` per skill per domain per bulan |
| 8 | Filter skill non-teknis | Hapus skill HR/Finance/Marketing |
| 9 | Validasi | 0 duplikat, 0 missing value |
| 10 | Simpan | `market_trend_history.csv` siap untuk Prophet/LSTM |

> **Catatan:** Kolom `is_tech_role` tidak digunakan karena tidak akurat (instruksi tim AI).  
> Filter IT dilakukan lewat **keyword matching pada `job_title`**.
